# 코호트: 4년 SCD 정답 + 최종 사용 환자 445명

**목적**: MUSIC 992명에서 Swee 2026 코호트 규칙으로 **730명**을 고르고 **4년 SCD 정답**을 계산한다. 그 정답을 팀 445명 분할 파일(`final_split_ecg_445.csv`, ML·DL 공통)에 붙여 **최종 사용 환자 445명 표**(`cohort_final.csv`)를 만든다. split은 팀 파일 그대로 쓴다. 02 전처리·증강(`02_ecg_preprocess_augment_445.ipynb`)은 이 445명 표를 쓴다.

| 출처 | 링크 |
|---|---|
| 논문 | Swee S, et al. medRxiv 2026. [본문](https://www.medrxiv.org/content/10.64898/2026.04.21.26351421v2.full) |
| 코드 (GPL-3.0, 규칙만 보고 다시 작성) | [github.com/stswee/multimodal_aecg_clinicalfeatures](https://github.com/stswee/multimodal_aecg_clinicalfeatures) (`2_Labeling/Labeling_MUSIC_Updated.ipynb`) |
| 팀 분할 파일 | `results/final_holter_ecg/final_split_ecg_445.csv` (`pid`, `split`, `SCD`). 445명, train 285 · val 71 · test 89 (ML·DL 공통) |

## 445명이 되기까지

| 단계 | 남은 환자 (SCD) | 뺀 환자 (SCD) | 뺀 이유 | 셀 |
|---|---|---|---|---|
| MUSIC 전체 | 992 | | | ① |
| 1. Holter 없음 | 936 (85) | 56 (6) | Holter 기록이 없어 ECG를 쓸 수 없음 | ② |
| 2. 비심장사 등 | 879 (85) | 57 (0) | 생존·SCD·PFD가 아닌 사망 원인 (SCD 여부를 판단할 수 없음) | ② |
| 3. 등록 전 이식형 심장기기 | 758 (71) | 121 (14) | 페이스메이커·CRT는 심전도 리듬을, ICD는 돌연사 발생을 바꿈 | ② |
| 4. 심장이식 | 746 (71) | 12 (0) | 이식으로 추적이 끝나 원래 심장의 4년 결과를 알 수 없음 | ② |
| 5. 4년 결과 모름 | **730** (71) | 16 (0) | 4년 전에 추적이 끝났고 사건도 없음 | ② |
| 6. 4년 안 PFD 사망 | 648 (71) | 82 (0) | SCD와 경쟁하는 사건이라 SCD 정답을 정할 수 없음 (정답 빈 값) | ③ |
| 7. 4년 뒤 PFD 사망 | 643 (71) | 5 (0) | 4년 정답은 0이지만, 팀 데이터는 PFD 사망을 시점과 상관없이 모두 뺌 | ⑤ |
| 8. 병인 코드 6·9, 나이 `>89` | 617 (71) | 26 (0) | 팀 임상 데이터에 이 값의 환자가 한 명도 없음 (병인 6: 심근경색 후 심근병증, 9: 기타 / 나이 `>89`는 숫자가 아닌 값) | ⑤ |
| 9. 결측치 | **445** (50) | 172 (21) | 팀 전처리에서 결측치가 있는 환자를 채우지 않고 뺌. 156명은 임상 변수 원본에 빈칸 (좌심실 수축기말 직경 68, 우심실 이완기말 직경 66, Pro-BNP 56 등), 16명은 원본 임상 변수에는 빈칸이 없고 팀 분할 단계(552명 → 508명)에서 결측으로 빠짐 | ⑤ |

- 1~5는 Swee 2026 코호트 규칙, 6은 Swee 4년 정답 규칙 그대로다. 7~9는 팀 임상 전처리·분할에서 빠진 환자로, 이 노트북을 정리할 때 `subject-info.csv`와 팀 파일(`clinical_clean.csv` 552명, `final_split.csv` 508명)로 따로 확인한 값이다 (셀 출력에는 없음).
- 코드는 1~6을 직접 계산하고(②③), 7~9는 팀 445명 분할 파일을 읽어 반영한다(⑤).
- 최종 **445명: SCD 50 · 4년 안 심장사 없음 395**. split은 팀이 이 445명 기준으로 나눈 train 285 (SCD 32) · val 71 (SCD 8) · test 89 (SCD 10), ML·DL 공통

## 4년 정답 (③)

| 순서 | 규칙 | 사용 열 |
|---|---|---|
| 1 | Holter 있음 | `Holter available` = 1 |
| 2 | 결과가 생존, SCD, 펌프부전 사망(PFD) 중 하나 (비심장사 제외) | `Cause of death` ∈ {0, 3, 6, 7} |
| 3 | 등록 전 이식형 심장기기 없음 | `Prior implantable device` = 0 |
| 4 | 심장이식으로 추적 종료되지 않음 | `Exit of the study` ≠ 2 |
| 5 | 4년 결과를 알 수 있음: 4년 이상 추적했거나, 4년 안에 SCD/PFD | `Follow-up period from enrollment (days)` |

- 4년 = **1,460일** (4 × 365)
- `SCD_4year_label`: 4년 안 SCD = 1, 심장사 없음 = 0, **4년 안 PFD = 빈 값** (SCD 분석에서 빠짐)
- `PFD_4year_label`: 반대로 4년 안 SCD = 빈 값. Swee 규칙을 따라 함께 계산하지만 우리 모델에는 쓰지 않는다
- `outcome_code`: 0 = 4년 안 심장사 없음, 3 = SCD, 6 = PFD
- 4년이 지난 뒤 SCD·PFD로 사망한 환자는 4년 정답이 0 (4년 동안은 사건 없음). Swee 코드도 같음 (`Follow-up ≤ 1,460일` 조건)

## 최종 사용 환자 (⑤)

1. 팀 445명 분할 파일(`final_split_ecg_445.csv`)을 읽는다. 환자마다 train / val / test가 적혀 있다
2. 기기 이식 환자가 섞여 있지 않은지 검사한다. 팀 파일은 이미 뺀 상태라 **0명이 정상**
3. 정답은 팀 파일의 `SCD` 열 대신 ③에서 계산한 **4년 정답**(`SCD_4year_label`)을 붙인다. 두 값이 다른 환자가 있으면 출력한다 (**없어야 정상**)
4. 두 파일을 저장한다
   - `cohort_final.csv`: 445명의 환자 ID, split, 4년 정답, 결과 코드 → 02가 처리할 환자 목록
   - `labels_445.csv`: 445명의 환자 ID와 4년 정답 → 02 ⑥이 결과 폴더로 복사

## 결정 기록 (사용자가 정한 것만)

| 날짜 | 결정 |
|---|---|
| 2026-10-26 | Swee 2026 코호트(730명, 4년 정답)를 그대로 사용 |
| 2026-10-27 | 팀 분할 파일(508명)에서 등록 전 기기 이식 환자 63명을 빼고 **445명(4년 SCD 50명)으로 확정**. 정답은 `SCD_4year_label` |
| 2026-10-27 | 증강과 전처리도 445명(`cohort_final.csv`)만 대상으로 함 |
| 2026-11-04 | 분할 파일을 팀 445명 분할(`final_split_ecg_445.csv`, ML·DL 공통, test 89명)로 변경. `labels_445.csv`도 ⑤에서 저장 |

In [ ]:
# ① 설정 + 임상 CSV 읽기: Swee와 같은 방식 (구분자 ';', 소수점 ',', 숫자 안의 탭을 '.'로)
from pathlib import Path

import pandas as pd

# 경로: 본인 환경에 맞게 바꿀 줄은 MUSIC_DIR 하나 (RESULT_DIR은 이 폴더 기준 상대경로)
MUSIC_DIR = Path(
    "/mnt/c/data/music"
)  # ← MUSIC 원본 폴더 (PhysioNet 구조 그대로: subject-info.csv, RECORDS, Holter_ECG/)
RESULT_DIR = Path("../results/final_holter_ecg")  # 팀 분할 파일을 두는 곳 + 이 노트북 결과

SUBJECT_CSV = MUSIC_DIR / "subject-info.csv"
SPLIT_PATH = (
    RESULT_DIR / "final_split_ecg_445.csv"
)  # 팀 445명 분할 (pid, split, SCD), ML·DL 공통 → 미리 넣어 둠
COHORT_PATH = RESULT_DIR / "cohort_4year.csv"  # 결과: 730명 4년 정답 표 (split 없음)
FLOW_PATH = RESULT_DIR / "flow_cohort.csv"  # 결과: ② 규칙마다 남은 환자 수 표 (992 → 730)
FINAL_PATH = RESULT_DIR / "cohort_final.csv"  # 결과: 최종 사용 환자 445명 (split + 4년 정답)
LABELS_PATH = RESULT_DIR / "labels_445.csv"  # 결과: 정답표 (pid, SCD_4year_label), 02 ⑥이 복사

HORIZON_DAYS = 4 * 365  # 4년 = 1,460일

# 사망 원인 코드 (subject-info_codes.csv)
SURVIVOR, SCD = 0, 3
PFD_CODES = [6, 7]
TRANSPLANT_EXIT = 2  # Exit of the study = 2 → 심장이식

subject = pd.read_csv(SUBJECT_CSV, sep=";", decimal=",", engine="python", na_values=["", "NA"])
subject = subject.replace(r"\t", ".", regex=True)
assert subject.shape == (992, 103), subject.shape  # Swee 코드의 확인 조건과 같음
print("임상 CSV:", subject.shape)

In [ ]:
# ② 코호트 5단계: 단계마다 몇 명이 남는지 표로 기록
follow_up = subject["Follow-up period from enrollment (days)"]
cause = subject["Cause of death"]
event_within_4y = cause.isin([SCD] + PFD_CODES) & follow_up.le(HORIZON_DAYS)

criteria = {
    "Holter 있음": subject["Holter available"].eq(1),
    "생존·SCD·PFD만 (비심장사 제외)": cause.isin([SURVIVOR, SCD] + PFD_CODES),
    "등록 전 기기 이식 없음": subject["Prior implantable device"].eq(0),
    "심장이식 아님": subject["Exit of the study"].ne(TRANSPLANT_EXIT)
    | subject["Exit of the study"].isna(),
    "4년 결과 확인 가능": follow_up.ge(HORIZON_DAYS) | event_within_4y,
}

keep = pd.Series(True, index=subject.index)
flow = [{"단계": "시작 (MUSIC 전체)", "남은 환자": len(subject), "제외": 0}]
for step, rule in criteria.items():
    before = int(keep.sum())
    keep &= rule.fillna(False)
    flow.append({"단계": step, "남은 환자": int(keep.sum()), "제외": before - int(keep.sum())})

flow = pd.DataFrame(flow)
display(flow)
cohort = subject.loc[keep].copy()

In [ ]:
# ③ 4년 정답: SCD와 PFD를 서로의 분석에서 빼는 방식 (경쟁 사건은 '0'이 아니라 빈 값)
scd_4y = cohort["Cause of death"].eq(SCD) & cohort["Follow-up period from enrollment (days)"].le(
    HORIZON_DAYS
)
pfd_4y = cohort["Cause of death"].isin(PFD_CODES) & cohort[
    "Follow-up period from enrollment (days)"
].le(HORIZON_DAYS)

labels = pd.DataFrame(
    {
        "pid": cohort["Patient ID"].str.strip().values,  # 'P0001' 형식 (Swee는 'P'를 뗀 '0001')
        "SCD_4year": scd_4y.astype(int).values,
        "PFD_4year": pfd_4y.astype(int).values,
    }
)
labels["SCD_4year_label"] = labels["SCD_4year"].where(labels["PFD_4year"].eq(0)).astype("Int64")
labels["PFD_4year_label"] = labels["PFD_4year"].where(labels["SCD_4year"].eq(0)).astype("Int64")
labels["outcome_code"] = 0  # 0 = 4년 안 심장사 없음
labels.loc[labels["SCD_4year"].eq(1), "outcome_code"] = 3
labels.loc[labels["PFD_4year"].eq(1), "outcome_code"] = 6

assert labels["pid"].is_unique and not (labels["SCD_4year"].eq(1) & labels["PFD_4year"].eq(1)).any()
print(
    "4년 결과:",
    labels["outcome_code"].map({0: "심장사 없음", 3: "SCD", 6: "PFD"}).value_counts().to_dict(),
)
print(
    "SCD 분석 대상:",
    labels["SCD_4year_label"].notna().sum(),
    "명 (SCD",
    int(labels["SCD_4year_label"].sum()),
    "명)",
)

In [ ]:
# ④ 저장: 730명 4년 정답 표(환자 1줄)와 ② 단계별 인원 표
labels.to_csv(COHORT_PATH, index=False)
flow.to_csv(FLOW_PATH, index=False)
print("저장:", COHORT_PATH, len(labels), "명")
print("저장:", FLOW_PATH)

In [ ]:
# ⑤ 최종 사용 환자: 팀 445명 분할 파일 + ③의 4년 정답 (기기 이식 환자가 없는지 확인)
split = pd.read_csv(SPLIT_PATH)
split["pid"] = split["pid"].str.strip()
device = pd.DataFrame(
    {"pid": subject["Patient ID"].str.strip(), "device": subject["Prior implantable device"]}
)
split = split.merge(device, on="pid", how="left", validate="one_to_one")
assert split["device"].notna().all(), "임상 CSV에 없는 환자가 분할 파일에 있음"

no_device = split["device"].eq(0)
print(
    "분할 파일:",
    len(split),
    "명 · 기기 이식 제외",
    int((~no_device).sum()),
    "명",
    split.loc[~no_device, "device"].value_counts().to_dict(),
)  # 코드: 1·2·3 = Pacemaker, 4·8 = CRT, 5 = ICD, 6 = 기타 기기

final = split.loc[no_device].merge(
    labels[["pid", "SCD_4year_label", "outcome_code"]], on="pid", how="left", validate="one_to_one"
)
assert final["SCD_4year_label"].notna().all(), "730명 코호트 밖 환자나 PFD 환자가 남아 있음"
changed = final[final["SCD"] != final["SCD_4year_label"]]
print(
    "정답이 바뀐 환자 (분할 파일 SCD → 4년 정답):",
    changed[["pid", "SCD", "SCD_4year_label"]].to_dict("records"),
)

final = final[["pid", "split", "SCD_4year_label", "outcome_code"]]
display(pd.crosstab(final["split"], final["SCD_4year_label"], margins=True))
final.to_csv(FINAL_PATH, index=False)
print("저장:", FINAL_PATH, len(final), "명 · SCD", int(final["SCD_4year_label"].sum()), "명")
final[["pid", "SCD_4year_label"]].to_csv(LABELS_PATH, index=False)
print("저장:", LABELS_PATH)

📝 **결과 메모**
- ② 단계별 인원 (기대: 992 → 936 → 879 → 758 → 746 → 730. Swee 논문: Holter 없음 56, 결과 범주 밖 57, 기기 121, 이식 12, 4년 불확실 16):
- ③ 4년 결과 (기대: 심장사 없음 577, SCD 71, PFD 82 → PFD를 뺀 648명이 SCD 정답 대상, ⑤에서 445명만 남김):
- ⑤ 분할 파일 인원 → 기기 이식 제외 → 최종 인원, split별 SCD (기대: 445 → 0 제외 → 445, SCD 50, train 285 · val 71 · test 89):